# Results: angular clustering of CEERS high-z galaxies

Collects every run of `run_pipeline.py` and shows:

1. Results table: N, $A_w$, $r_0$, $b_g$, $M_h$ per redshift bin (saved as CSV and LaTeX)
2. $w(\theta)$ with the fitted power law, one panel per bin
3. $A_w$, $r_0$, $b_g$ and $\log M_h$ against redshift
4. Galaxy bias against halo-bias tracks of fixed mass
5. Luminosity function used (`inputs/lf_table.csv`)
6. Completeness $C(M_{UV}, z)$ per z-bin (from the sky-map mocks)
7. Data and random positions on the sky
8. Run logs

Results come from `summary/<z_bin>/summary_*.yaml` (z-bins = `survey.z_bins` in `parameters.yaml`,
e.g. `z7p5_9p5`). Only runs of the current pipeline (summaries with a `z_bin` key) are read; the old
`summary/z8.00/` … folders are ignored. A bin can have several runs (one per settings hash), so by default
the **newest run per (z_bin, M_UV bin)** is used. To use another run, put its tag in `PIN` in the next cell.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
from IPython.display import display

# project root = first parent folder that has summary/ and codes/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "summary").is_dir() and (p / "codes").is_dir())
sys.path.insert(0, str(ROOT / "codes"))
from run_pipeline import make_phi, load_lf_table, load_completeness
from halo_mass import halo_bias
from acf_estimator import _linear_growth_factor
from astropy.cosmology import FlatLambdaCDM

with open(ROOT / "codes" / "parameters.yaml") as f:
    CFG = yaml.safe_load(f)
COSMO = CFG["cosmology"]
BETA = CFG["acf"]["beta"]
cosmo = FlatLambdaCDM(H0=COSMO["H0"], Om0=COSMO["Om0"])

PIN = {}            # e.g. {("z7p5_9p5", (-23.0, -19.5)): "z8.50_seed42_534b4aa6_M-23.0_-19.5"}
MAPS = ROOT / "inputs/depth_maps/CEERS_SkyCompleteness_zbins/CEERS_SkyCompleteness_zbins"   # sky-map folders
DETECTION_SIGMA = 2.0   # A_w / A_w_err below this -> shown as an upper limit

# categorical colours, assigned to bins in a fixed order (never cycled)
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
INK, MUTED, GRID = "#1f1f1e", "#6b6a64", "#e4e3de"
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "lines.linewidth": 2,
    "lines.markersize": 7, "legend.frameon": False, "font.size": 10,
})
print("project:", ROOT)

## 0. Latest run

Everything from the most recent `run_pipeline.py` call: the run log, the results, and the (\theta)$ table and plot of each M bin.
Results written within `RUN_WINDOW_MIN` minutes of the newest summary count as the same run (a run over several z-bins writes several).


In [ ]:
RUN_WINDOW_MIN = 30     # summaries written within this many minutes of the newest one count as the same run

summ = []
for f in (ROOT / "summary").glob("z*/summary_*.yaml"):
    with open(f) as fh:
        s = yaml.safe_load(fh)
    if "z_bin" in s:                                  # current pipeline only
        s.update(tag=f.stem.removeprefix("summary_"), mtime=f.stat().st_mtime)
        summ.append(s)
newest = max(s["mtime"] for s in summ)
last = sorted((s for s in summ if newest - s["mtime"] <= RUN_WINDOW_MIN * 60),
              key=lambda s: (s["z_bin"], tuple(s["M_UV_bin"])))
when = pd.Timestamp.fromtimestamp(newest).strftime("%Y-%m-%d %H:%M")      # local time
print(f"latest run: {when}  ({len(last)} result(s), z-bins: {', '.join(sorted({s['z_bin'] for s in last}))})\n")

# 1. run logs (one per z-bin; overwritten by each run, so this is the latest one)
for s in {s["z_bin"]: s for s in last}.values():
    log = ROOT / "summary" / s["z_bin"] / f"run_log_z{s['z_drop']:.2f}.txt"
    print(f"===== log: {s['z_bin']}  ({log.name})")
    print(log.read_text() if log.exists() else "  no log\n")

# 2. results table
def mh_str(s):
    v, lo, hi = (s.get(k, np.nan) for k in ("log10_Mh_h_inv_Msun", "log10_Mh_lo", "log10_Mh_hi"))
    return "--" if not np.isfinite(v) else f"{v:.2f} [{lo:.2f}, {hi:.2f}]"

table = pd.DataFrame([{
    "z_bin": s["z_bin"], "z_a": f"{s['z_a_range'][0]:g}-{s['z_a_range'][1]:g}",
    "M_UV": f"[{s['M_UV_bin'][0]:g}, {s['M_UV_bin'][1]:g})", "z_eff": round(s["z_eff"], 2),
    "N": s["n_data"], "N_rand": s["n_random"],
    "theta": f"{s['theta_bins_arcsec'][0]:g}-{s['theta_bins_arcsec'][1]:g}\" ({s['theta_bins_arcsec'][2]} bins)",
    "A_w": f"{s['A_w']:.3f} ± {s['A_w_err']:.3f}", "S/N": round(s["A_w"] / s["A_w_err"], 2),
    "r0 (h^-1 Mpc)": f"{s['r0_h_inv_mpc']:.2f} ± {s['r0_err_h_inv_mpc']:.2f}",
    "b_g": f"{s['bias']:.2f} ± {s['bias_err']:.2f}", "log M_h (h^-1 Msun)": mh_str(s), "tag": s["tag"],
} for s in last])
display(table)

# 3. w(theta) per result: table + plot
min_rr, min_dd = CFG["acf"]["min_RR_counts"], CFG["acf"].get("min_DD_counts", 0)   # current settings
ncol = min(3, len(last))
nrow = int(np.ceil(len(last) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4.4 * ncol, 3.4 * nrow), squeeze=False)
for i, (ax, s) in enumerate(zip(axes.flat, last)):
    acf = pd.read_csv(ROOT / "outputs" / s["z_bin"] / f"acf_results_{s['tag']}.csv")
    used = (acf["RR"] >= min_rr) & (acf["DD"] >= min_dd)
    print(f"\n===== w(theta): {s['tag']}")
    display(acf.assign(used=used).round(4))

    c = SERIES[i % len(SERIES)]
    ax.errorbar(acf.theta_arcsec[used], acf.w_obs[used], acf.w_err[used], fmt="o", color=c, capsize=0, lw=1.2)
    ax.errorbar(acf.theta_arcsec[~used], acf.w_obs[~used], acf.w_err[~used], fmt="o", mfc="white", color=c, lw=1.2)
    A_raw = s.get("A_w_raw", s["A_w"])
    th = np.geomspace(acf.theta_lo.min(), acf.theta_hi.max(), 200)
    ax.plot(th, A_raw * (th ** -BETA - s["ic_over_Aw"]), color=INK, lw=1.5)
    ax.axhline(0, color=MUTED, lw=0.8)
    ax.set_xscale("log")
    ax.set_xlabel(r"$\theta$ (arcsec)")
    ax.set_ylabel(r"$w(\theta)$")
    ax.set_title(f"{s['z_bin']}, M [{s['M_UV_bin'][0]:g}, {s['M_UV_bin'][1]:g})\n"
                 f"N = {s['n_data']}   $A_w$ = {s['A_w']:.2f} ± {s['A_w_err']:.2f}", fontsize=9, color=INK)
for ax in axes.flat[len(last):]:
    ax.set_visible(False)
fig.tight_layout()
plt.show()

## 1. Results table

In [ ]:
rows = []
for f in sorted((ROOT / "summary").glob("z*/summary_*.yaml")):
    with open(f) as fh:
        s = yaml.safe_load(fh)
    if "z_bin" not in s:
        continue                    # run of the old (depth-map, per-z_drop) pipeline
    s["tag"] = f.stem.removeprefix("summary_")
    s["file"] = f
    s["mtime"] = f.stat().st_mtime
    rows.append(s)
runs = pd.DataFrame(rows)
runs["M_bin"] = runs["M_UV_bin"].map(tuple)
runs["theta_min"] = runs["theta_bins_arcsec"].map(lambda t: t[0])
runs["theta_max"] = runs["theta_bins_arcsec"].map(lambda t: t[1])
for col in ["log10_Mh_h_inv_Msun", "log10_Mh_lo", "log10_Mh_hi", "A_w_raw", "contamination_fraction"]:
    if col not in runs:
        runs[col] = np.nan          # runs made before these fields existed

def pick(key, group):
    """index of the run to use for this (z_bin, M_bin): the pinned tag, else the newest"""
    if key in PIN:
        return group.index[group["tag"] == PIN[key]][0]
    return group["mtime"].idxmax()

chosen = [pick(key, group) for key, group in runs.groupby(["z_bin", "M_bin"])]
res = runs.loc[chosen].sort_values(["z_bin", "M_bin"]).reset_index(drop=True)   # bright M bin first
res["label"] = [f"{r.z_bin}, M {r.M_bin[0]:g} to {r.M_bin[1]:g}" for r in res.itertuples()]
res["snr"] = res["A_w"] / res["A_w_err"]
res["detected"] = res["snr"] >= DETECTION_SIGMA
res["A_w_upper"] = np.where(res["detected"], np.nan, np.maximum(res["A_w"], 0) + DETECTION_SIGMA * res["A_w_err"])
res["colour"] = [SERIES[i % len(SERIES)] for i in range(len(res))]

print(f"{len(runs)} runs found, {len(res)} used (newest per bin unless pinned)")
show = res[["z_bin", "z_a_range", "z_eff", "M_bin", "n_data", "theta_min", "theta_max", "A_w", "A_w_err", "snr",
            "r0_h_inv_mpc", "r0_err_h_inv_mpc", "bias", "bias_err",
            "log10_Mh_h_inv_Msun", "log10_Mh_lo", "log10_Mh_hi", "tag"]]
display(show.round(3))

In [ ]:
# all runs, to see how results moved between settings
display(runs.sort_values(["z_bin", "mtime"])[["z_bin", "z_eff", "M_bin", "n_data", "theta_min",
                                                 "A_w", "A_w_err", "r0_h_inv_mpc", "bias", "tag"]].round(3))

In [ ]:
def pm(v, e, nd=3):
    return "--" if not np.isfinite(v) else f"${v:.{nd}f}\\pm{e:.{nd}f}$"

def mh(r):
    if not np.isfinite(r.log10_Mh_h_inv_Msun):
        return "--"
    lo = r.log10_Mh_h_inv_Msun - r.log10_Mh_lo if np.isfinite(r.log10_Mh_lo) else np.nan
    hi = r.log10_Mh_hi - r.log10_Mh_h_inv_Msun
    lo_s = f"{lo:.2f}" if np.isfinite(lo) else "?"
    return f"${r.log10_Mh_h_inv_Msun:.2f}^{{+{hi:.2f}}}_{{-{lo_s}}}$"

table = pd.DataFrame({
    "$z_a$ range": [f"{lo:g}--{hi:g}" for lo, hi in res["z_a_range"]],
    "$M_{UV}$": [f"[{lo:g}, {hi:g})" for lo, hi in res["M_bin"]],
    "$z_{\\rm eff}$": res["z_eff"].round(2),
    "$N$": res["n_data"],
    "$A_w$": [pm(a, e) if d else f"$<{u:.2f}$" for a, e, d, u in zip(res.A_w, res.A_w_err, res.detected, res.A_w_upper)],
    "$r_0$ ($h^{-1}$ Mpc)": [pm(a, e) for a, e in zip(res.r0_h_inv_mpc, res.r0_err_h_inv_mpc)],
    "$b_g$": [pm(a, e) for a, e in zip(res.bias, res.bias_err)],
    "$\\log_{10} M_h$ ($h^{-1}M_\\odot$)": [mh(r) for r in res.itertuples()],
})
display(table)

out_csv = ROOT / "summary" / "results_table.csv"
res.drop(columns=["file", "colour", "label"]).to_csv(out_csv, index=False)
try:
    latex = table.to_latex(index=False, escape=False, column_format="r" * table.shape[1])
except ImportError:                     # pandas needs jinja2 for to_latex: write a plain booktabs table instead
    latex = ("\\begin{tabular}{" + "r" * table.shape[1] + "}\n\\toprule\n" + " & ".join(table.columns) + " \\\\\n\\midrule\n"
             + "".join(" & ".join(map(str, row)) + " \\\\\n" for row in table.itertuples(index=False))
             + "\\bottomrule\n\\end{tabular}\n")
(ROOT / "summary" / "results_table.tex").write_text(latex)
print(latex)
print("saved", out_csv, "and results_table.tex")

## 2. $w(\theta)$ per bin

Points: Landy–Szalay $w(\theta)$ with bootstrap errors. Line: fitted $A_w(\theta^{-\beta} - \mathrm{IC}/A_w)$
(raw amplitude, i.e. before any contamination correction, since the points are raw too).
Open points: bins left out of the fit (RR below `min_RR_counts` or DD below `min_DD_counts`).
Solid line: fitted range; dotted: extrapolation. Saved as PDF (vector, for the paper) and 300-dpi PNG.


In [ ]:
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D

# journal style, applied to this figure only (rc_context), so the notebook's screen style is untouched
TEXTWIDTH_IN = 7.1      # two-column text width in inches: ApJ/AAS 7.1, MNRAS 6.97, A&A 7.08
PAPER_RC = {
    "font.family": "serif", "font.serif": ["STIXGeneral", "DejaVu Serif"], "mathtext.fontset": "stix",
    "font.size": 8, "axes.labelsize": 9, "legend.fontsize": 7.5, "xtick.labelsize": 7.5, "ytick.labelsize": 7.5,
    "axes.grid": False, "axes.spines.top": True, "axes.spines.right": True,
    "axes.edgecolor": INK, "axes.labelcolor": INK, "axes.linewidth": 0.6,
    "xtick.color": INK, "ytick.color": INK, "xtick.direction": "in", "ytick.direction": "in",
    "xtick.top": True, "ytick.right": True, "xtick.minor.visible": True, "ytick.minor.visible": True,
    "xtick.major.size": 3.5, "ytick.major.size": 3.5, "xtick.minor.size": 2, "ytick.minor.size": 2,
    "xtick.major.width": 0.6, "ytick.major.width": 0.6, "xtick.minor.width": 0.5, "ytick.minor.width": 0.5,
    "lines.markersize": 4, "errorbar.capsize": 0,
    "savefig.dpi": 300, "pdf.fonttype": 42, "ps.fonttype": 42,   # embed TrueType fonts (journal requirement)
}


def bin_text(i, r):
    """panel annotation: letter, selection window, z_eff, N and the amplitude (or its upper limit)"""
    za_lo, za_hi = r.z_a_range
    m_lo, m_hi = r.M_bin
    amp = (rf"$A_w = {r.A_w:.2f} \pm {r.A_w_err:.2f}$" if r.detected
           else rf"$A_w < {r.A_w_upper:.2f}$ ({DETECTION_SIGMA:g}$\sigma$)")
    return "\n".join([
        rf"({chr(97 + i)})  ${za_lo:g} \leq z_a < {za_hi:g}$",
        rf"${m_lo:g} \leq M_{{\rm UV}} < {m_hi:g}$",
        rf"$z_{{\rm eff}} = {r.z_eff:.2f}$,  $N = {r.n_data}$",
        amp,
    ])


n = len(res)
ncol = min(3, n)
nrow = int(np.ceil(n / ncol))
min_rr, min_dd = CFG["acf"]["min_RR_counts"], CFG["acf"].get("min_DD_counts", 0)

with plt.rc_context(PAPER_RC):
    fig, axes = plt.subplots(nrow, ncol, figsize=(TEXTWIDTH_IN * ncol / 3, 2.3 * nrow + 0.4),
                             sharex=True, squeeze=False, layout="constrained")

    for i, (ax, r) in enumerate(zip(axes.flat, res.itertuples())):
        acf = pd.read_csv(ROOT / "outputs" / r.z_bin / f"acf_results_{r.tag}.csv")
        used = (acf["RR"] >= min_rr) & (acf["DD"] >= min_dd)

        pts = dict(fmt="o", color=r.colour, mec=r.colour, mew=0.8, elinewidth=0.8, zorder=3)
        ax.errorbar(acf.theta_arcsec[used], acf.w_obs[used], acf.w_err[used], mfc=r.colour, **pts)
        ax.errorbar(acf.theta_arcsec[~used], acf.w_obs[~used], acf.w_err[~used], mfc="white", **pts)

        # model: solid over the fitted angular range, dotted where it is extrapolated
        A_raw = r.A_w_raw if np.isfinite(r.A_w_raw) else r.A_w
        th = np.geomspace(acf.theta_lo.min(), acf.theta_hi.max(), 300)
        model = A_raw * (th ** -BETA - r.ic_over_Aw)
        in_fit = (th >= acf.theta_lo[used].min()) & (th <= acf.theta_hi[used].max())
        ax.plot(th, model, color=INK, lw=0.8, ls=":", zorder=2)
        ax.plot(th[in_fit], model[in_fit], color=INK, lw=1.1, zorder=2)

        ax.axhline(0, color=MUTED, lw=0.5, ls="--", zorder=1)
        ax.set_xscale("log")
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:g}"))
        # y-range from the data + errors; keep the top ~38% of the panel free for the annotation
        lo = np.nanmin(acf.w_obs - acf.w_err)
        hi = max(np.nanmax(acf.w_obs + acf.w_err), np.nanmax(model[in_fit]))
        pad = 0.05 * (hi - lo)
        ax.set_ylim(lo - pad, lo - pad + (hi - lo + 2 * pad) / 0.62)
        ax.text(0.96, 0.95, bin_text(i, r), transform=ax.transAxes, ha="right", va="top",
                fontsize=7, linespacing=1.4, color=INK)

    # empty slots: hide them and put tick labels back on the panel above (sharex removed them)
    for j in range(n, nrow * ncol):
        axes.flat[j].set_visible(False)
        if j - ncol >= 0:
            axes.flat[j - ncol].tick_params(labelbottom=True)

    fig.supxlabel(r"$\theta$ [arcsec]", fontsize=9)
    fig.supylabel(r"$w(\theta)$", fontsize=9)
    key = [
        Line2D([], [], marker="o", ls="", color=MUTED, mfc=MUTED, label="used in fit"),
        Line2D([], [], marker="o", ls="", color=MUTED, mfc="white", label="excluded (low RR/DD)"),
        Line2D([], [], color=INK, lw=1.1, label=rf"$A_w(\theta^{{-{BETA:g}}} - C)$ fit"),
        Line2D([], [], color=INK, lw=0.8, ls=":", label="extrapolation"),
    ]
    fig.legend(handles=key, loc="outside upper center", ncol=4, frameon=False, handlelength=1.8, columnspacing=1.5)

    out = ROOT / "summary" / "wtheta_all_bins"
    fig.savefig(out.with_suffix(".pdf"))                  # vector, for the paper
    fig.savefig(out.with_suffix(".png"), dpi=300)         # raster, for slides / quick look
    plt.show()

## 3. Results against redshift

One quantity per panel (no shared y-axes). Filled points: detections ($A_w/\sigma \geq$ `DETECTION_SIGMA`).
Arrows: upper limits on $A_w$ at that significance. $r_0$, $b_g$ and $M_h$ exist only where $A_w > 0$.

In [ ]:
quantities = [
    ("A_w", "A_w_err", r"$A_w$ (arcsec$^{\beta}$)"),
    ("r0_h_inv_mpc", "r0_err_h_inv_mpc", r"$r_0$ ($h^{-1}$ Mpc)"),
    ("bias", "bias_err", r"galaxy bias $b_g$"),
    ("log10_Mh_h_inv_Msun", None, r"$\log_{10} M_h$ ($h^{-1} M_\odot$)"),
]
fig, axes = plt.subplots(2, 2, figsize=(10, 7.5))
for ax, (col, ecol, label) in zip(axes.flat, quantities):
    for r in res.itertuples():
        v = getattr(r, col)
        if col == "A_w" and not r.detected:
            ax.errorbar(r.z_eff, r.A_w_upper, yerr=0.25 * r.A_w_upper, uplims=True, fmt="v", color=r.colour, mfc="white")
            continue
        if not np.isfinite(v):
            continue
        if ecol:
            yerr = getattr(r, ecol)
        else:                                   # asymmetric mass range; NaN lower end = unconstrained
            lo = v - r.log10_Mh_lo if np.isfinite(r.log10_Mh_lo) else 0.0
            yerr = [[lo], [r.log10_Mh_hi - v]]
        ax.errorbar(r.z_eff, v, yerr=yerr, fmt="o", color=r.colour, capsize=0, lw=1.5,
                    mfc=r.colour if r.detected else "white")
        if ecol is None and not np.isfinite(r.log10_Mh_lo):
            ax.annotate("", (r.z_eff, v - 0.8), (r.z_eff, v), arrowprops=dict(arrowstyle="->", color=r.colour))
    ax.set_xlabel("redshift $z_{\\rm eff}$")
    ax.set_ylabel(label)
axes[0, 0].axhline(0, color=MUTED, lw=0.8)
handles = [plt.Line2D([], [], marker="o", ls="", color=r.colour, label=r.label) for r in res.itertuples()]
fig.legend(handles=handles, loc="upper center", ncol=min(len(handles), 2), bbox_to_anchor=(0.5, 1.06))
fig.tight_layout()
fig.savefig(ROOT / "summary" / "results_vs_z.png", bbox_inches="tight")
plt.show()

## 4. Galaxy bias against halo-bias tracks

Lines: $b_h(M, z)$ of halos of fixed mass (Tinker et al. 2010, or the model in `parameters.yaml`),
computed with `halo_mass.py`. A galaxy point lying on a track has that effective halo mass.

In [ ]:
cosmo_pars = dict(Om0=COSMO["Om0"], Ob0=COSMO["Ob0"], h=COSMO["h"], ns=COSMO["ns"], sigma8_0=COSMO["sigma8_0"])
model = CFG.get("halo", {}).get("bias_model", "tinker10")
z_track = np.linspace(max(res.z_eff.min() - 1, 0.5), res.z_eff.max() + 1, 25)
growth = np.array([_linear_growth_factor(z, cosmo) for z in z_track])
track_masses = [9, 10, 11, 12]
greys = ["#c3c2b7", "#9a998f", "#6b6a64", "#3d3c38"]

fig, ax = plt.subplots(figsize=(6.5, 4.5))
for logM, g in zip(track_masses, greys):
    b = [halo_bias(10.0 ** logM, D, cosmo_pars, model)[0] for D in growth]
    ax.plot(z_track, b, color=g, lw=1.5)
    ax.text(z_track[-1], b[-1], f"  $10^{{{logM}}}$", color=g, va="center", fontsize=9)
for r in res.itertuples():
    if np.isfinite(r.bias):
        ax.errorbar(r.z_eff, r.bias, r.bias_err, fmt="o", color=r.colour, lw=1.5, capsize=0,
                    mfc=r.colour if r.detected else "white", label=r.label)
ax.set_yscale("log")
ax.set_xlabel("redshift")
ax.set_ylabel("bias")
ax.set_title(f"halo-bias tracks: {model}, masses in $h^{{-1}} M_\\odot$", fontsize=10, color=INK)
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(ROOT / "summary" / "bias_vs_halo_tracks.png", bbox_inches="tight")
plt.show()

## 5. Luminosity function used

In [ ]:
lf = load_lf_table(ROOT / CFG["paths"]["lf_table"])        # inputs/lf/lf_table.csv
display(lf)

styles = {"real": dict(marker="o", mfc=SERIES[0]), "interpolated": dict(marker="s", mfc="white"),
          "extrapolated": dict(marker="^", mfc="white")}
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, p, label in zip(axes, ["M_star", "log_phi_star", "alpha"],
                        [r"$M^*$ (AB)", r"$\log_{10}\phi^*$ (Mpc$^{-3}$ mag$^{-1}$)", r"$\alpha$"]):
    for kind, st in styles.items():
        t = lf[lf["type"] == kind]
        lo, hi = (t.get(f"{p}_err_lo"), t.get(f"{p}_err_hi"))
        yerr = None if lo is None else [lo.fillna(0), hi.fillna(0)]
        ax.errorbar(t.z, t[p], yerr=yerr, ls="", color=SERIES[0], lw=1.2, capsize=0, label=kind, **st)
    ax.plot(lf.z, lf[p], color=SERIES[0], lw=1, alpha=0.4)
    for r in res.itertuples():
        ax.axvline(r.z_eff, color=r.colour, lw=0.8, ls=":")
    ax.set_xlabel("z")
    ax.set_ylabel(label)
axes[0].invert_yaxis()
axes[0].legend()
fig.suptitle("Schechter parameters (dotted: z_eff of the result bins)", color=INK, fontsize=10)
fig.tight_layout()
fig.savefig(ROOT / "summary" / "lf_parameters.png", bbox_inches="tight")
plt.show()

In [ ]:
phi = make_phi(lf)
M = np.linspace(-23.5, -16, 200)
fig, ax = plt.subplots(figsize=(6.5, 4.5))
for r in res.drop_duplicates("z_bin").itertuples():
    ax.plot(M, phi(M, r.z_drop), color=r.colour, label=f"{r.z_bin} (z = {r.z_drop:g})")
for lo_hi in sorted(set(res.M_bin)):
    ax.axvspan(*lo_hi, color=GRID, alpha=0.35, lw=0)
ax.set_yscale("log")
ax.set_xlabel(r"$M_{\rm UV}$ (AB)")
ax.set_ylabel(r"$\phi$ (Mpc$^{-3}$ mag$^{-1}$)")
ax.invert_xaxis()
ax.legend()
ax.set_title("UV LF at the centre of each z-bin (shaded: M_UV bins used)", fontsize=10, color=INK)
fig.tight_layout()
fig.savefig(ROOT / "summary" / "lf_curves.png", bbox_inches="tight")
plt.show()

In [ ]:
# UV luminosity functions of the rows of inputs/lf/lf_table.csv, one curve per redshift
Z_SWITCH = 8.5          # Bouwens+21 below, the z > 8.5 LF from here on (as in make_lf_table.py)
LF_ROWS = ["real"]      # row types to draw: "real", "interpolated", "extrapolated"

lf = load_lf_table(ROOT / CFG["paths"]["lf_table"])        # inputs/lf/lf_table.csv
phi = make_phi(lf)
rows = lf[lf["type"].isin(LF_ROWS)]
M = np.linspace(-23.0, -16.0, 200)
dashes = {"real": "-", "interpolated": "--", "extrapolated": ":"}

panels = [(rows[rows.z < Z_SWITCH], plt.cm.Blues, rf"$z < {Z_SWITCH:g}$: Bouwens+21"),
          (rows[rows.z >= Z_SWITCH], plt.cm.Oranges, rf"$z \geq {Z_SWITCH:g}$: $M^*$ and $\alpha$ fixed")]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), sharex=True, sharey=True)
for ax, (t, cmap, title) in zip(axes, panels):
    shades = cmap(np.linspace(0.35, 1.0, len(t)))          # one hue per source, darker = higher z
    for r, shade in zip(t.itertuples(), shades):
        ax.plot(M, phi(M, r.z), color=shade, ls=dashes[r.type], label=f"z = {r.z:g}")
    ax.set_yscale("log")
    ax.set_xlabel(r"$M_{\rm UV}$ (AB)")
    ax.set_title(title, fontsize=10, color=INK)
    ax.legend(ncol=2, loc="lower left")
axes[0].set_ylabel(r"$\phi$ (Mpc$^{-3}$ mag$^{-1}$)")
axes[0].set_ylim(1e-8, 1e-1)
axes[0].invert_xaxis()                                     # bright end on the right (shared axis)
fig.suptitle(f"UV luminosity functions in {Path(CFG['paths']['lf_table']).name} ({', '.join(LF_ROWS)} rows)",
             color=INK, fontsize=10)
fig.tight_layout()
fig.savefig(ROOT / "summary" / "lf_all_redshifts.png", bbox_inches="tight")
plt.show()

## 6. Completeness per z-bin

$C(M_{UV}, z_{\rm true})$ as used by the pipeline (`load_completeness`: fraction of the sky-map mocks passing layer `injection.layer`, footprint average, monotone in $M_{UV}$), at a few true redshifts per z-bin.

In [ ]:
layer = CFG["injection"].get("layer", 4)
zbins = list(res.drop_duplicates("z_bin").itertuples())
fig, axes = plt.subplots(1, len(zbins), figsize=(6 * len(zbins), 4.2), squeeze=False)
M = np.linspace(-23.0, -17.0, 200)
for ax, r in zip(axes.flat, zbins):
    C, _ = load_completeness(MAPS / r.z_bin / f"ceers_skycompleteness_mocks_{r.z_bin}.fits", cosmo,
                             CFG["selection"].get("monotone_completeness", True), layer=layer)
    lo, hi = r.z_true_range
    for i, z in enumerate(np.linspace(lo + 0.1 * (hi - lo), hi - 0.1 * (hi - lo), 5)):
        ax.plot(M, C(M, z), color=SERIES[i], label=f"z_true = {z:.2f}")
    for b in res[res.z_bin == r.z_bin].M_bin:
        ax.axvspan(*b, color=GRID, alpha=0.3, lw=0)
    ax.set_xlabel(r"$M_{\rm UV}$ (AB)")
    ax.set_ylabel(f"completeness C (layer {layer})")
    ax.set_title(f"{r.z_bin}: z_a window {r.z_a_range[0]:g}-{r.z_a_range[1]:g}", fontsize=10, color=INK)
    ax.invert_xaxis()
    ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(ROOT / "summary" / "completeness.png", bbox_inches="tight")
plt.show()


## 7. Data and randoms on the sky

Data: the vetted catalogue (`paths.real_catalog`) in the bin's `z_a` window, after `survey.data_cuts` from `parameters.yaml` (the cuts as they are **now**,
which may differ from those of an older run). Randoms: the cached catalogue of the run used.

In [ ]:
n = len(res)
ncol = min(2, n)
nrow = int(np.ceil(n / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(7 * ncol, 3.2 * nrow), squeeze=False)
sv = CFG["survey"]
from astropy.table import Table
cat = Table.read(ROOT / CFG["paths"]["real_catalog"]).to_pandas()

for ax, r in zip(axes.flat, res.itertuples()):
    za_lo, za_hi = r.z_a_range
    data = cat[(cat[sv["z_col"]] >= za_lo) & (cat[sv["z_col"]] < za_hi)]
    for cut in sv.get("data_cuts") or []:
        data = data.query(cut)
    lo, hi = r.M_bin
    data = data[(data[sv["muv_col"]] >= lo) & (data[sv["muv_col"]] < hi)]
    rf = ROOT / "random_catalogs" / f"random_catalog_{r.tag}.csv"
    if rf.exists():
        rand = pd.read_csv(rf)
        ax.scatter(rand.ra, rand.dec, s=0.3, color=MUTED, alpha=0.25, rasterized=True, label=f"randoms ({len(rand)})")
    else:
        print("random catalogue not found:", rf.name)
    ax.scatter(data[sv["ra_col"]], data[sv["dec_col"]], s=10, color=r.colour, edgecolor="white", lw=0.4,
               label=f"data ({len(data)})")
    ax.invert_xaxis()
    ax.set_aspect(1 / np.cos(np.radians(data[sv["dec_col"]].mean())))
    ax.set_xlabel("RA (deg)")
    ax.set_ylabel("Dec (deg)")
    ax.set_title(r.label, fontsize=10, color=INK)
    ax.legend(loc="upper right", markerscale=3, fontsize=8)
for ax in axes.flat[n:]:
    ax.set_visible(False)
fig.tight_layout()
fig.savefig(ROOT / "summary" / "sky_data_randoms.png", bbox_inches="tight", dpi=150)
plt.show()

## 8. Run logs (last run of each bin)

In [ ]:
for r in res.drop_duplicates("z_bin").itertuples():
    log = ROOT / "summary" / r.z_bin / f"run_log_z{r.z_drop:.2f}.txt"
    print(f"===== {r.z_bin}  ({log.name})")
    print(log.read_text() if log.exists() else "  no log\n")
